In [81]:
import ee
import earthaccess
import os
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Configuration from your .ipynb

BBOX = [8.8, 55.8, 9.4, 56.1]; 

PATCH_SIZE = 13
START_DATE = "2015-01-01"
END_DATE = "2018-03-1"
PATCH_CENTER = PATCH_SIZE // 2
AOI = ee.Geometry.Rectangle(BBOX)

In [82]:
def load_ismn(file_path):
    with open(file_path, 'r') as f:
        header = f.readline().strip().split()

    try:
        network, station = header[1], header[2]
        lat, lon = float(header[3]), float(header[4])
    except (IndexError, ValueError):
        return None

    df = pd.read_csv(
        file_path, 
        sep=r'\s+', 
        skiprows=1, 
        header=None, 
        engine='python'
    )

    if df.shape[1] < 4:
        return None

    df = df.iloc[:, :4]
    df.columns = ['date', 'time', 'sm', 'flag']

    # Filter for Good quality
    df = df[df['flag'] == 'G']
    if df.empty: return None

    # Cleaning
    df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df['sm'] = pd.to_numeric(df['sm'], errors='coerce')
    df = df.dropna(subset=['sm'])

    # Unit conversion (Percentage to m3/m3)
    if df['sm'].mean() > 1.0:
        df['sm'] = df['sm'] / 100.0

    df['point_id'] = f"{network}_{station}"
    df['latitude'] = lat
    df['longitude'] = lon

    return df[['point_id', 'timestamp', 'sm', 'latitude', 'longitude']]

# --- EXECUTION ---
stm_files = glob.glob('Data/Denmark/HOBE/*/*.stm', recursive=True)

dfs = [load_ismn(f) for f in stm_files]
dfs = [d for d in dfs if d is not None]
df_combined = pd.concat(dfs, ignore_index=True)

# 1. MULTI-SENSOR COLLAPSE (Handle A, B, C duplicates)
# We group by timestamp to average the redundant sensors at each specific hour/minute
ground_truth = df_combined.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()

# 2. DAILY AGGREGATION
# Now we move from hourly/sub-hourly to daily
ground_truth['timestamp'] = (
    ground_truth['timestamp']
    .dt.tz_localize(
        'Europe/Copenhagen',
        nonexistent='shift_forward',
        ambiguous='NaT'
    )
    .dropna()
    .dt.tz_convert('UTC')
    .dt.floor('D')
)

ground_truth = ground_truth.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()
ground_truth = ground_truth[
    ground_truth['timestamp'] >= '2015-01-01' 
]


print(f"Final ground truth shape: {ground_truth.shape}")
print(f"Unique stations: {ground_truth['point_id'].nunique()}")

Final ground truth shape: (20981, 5)
Unique stations: 30


In [83]:
# ================= CONFIG =================
TARGET_SCALE = 1000
TARGET_CRS = 'EPSG:32632'
PATCH_RADIUS = (PATCH_SIZE - 1) // 2


# ================= DATA LOADERS =================

def get_s2(aoi, start, end):
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(aoi) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

    def add_idx(img):
        ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
        ndwi = img.normalizedDifference(['B8','B11']).rename('NDWI')
        return ee.Image.cat([ndvi, ndwi]).float() \
            .copyProperties(img, ['system:time_start'])

    return s2.map(add_idx)


def get_s1(aoi, start, end):
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(aoi) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    def proc(img):
        bands = img.bandNames()

        vv = ee.Image(
            ee.Algorithms.If(bands.contains('VV'), img.select('VV'), ee.Image.constant(-20))
        )

        vh = ee.Image(
            ee.Algorithms.If(bands.contains('VH'), img.select('VH'), vv)
        )

        angle = img.select('angle')  # always present in GRD

        vv = vv.toFloat().unmask(-20)
        vh = vh.toFloat().unmask(-20)
        angle = angle.toFloat().unmask(35)  # typical mid-angle fallback

        ratio = vv.subtract(vh).rename('VV_VH_Ratio')

        return ee.Image.cat([
            vv.rename('VV'),
            vh.rename('VH'),
            ratio,
            angle.rename('angle')
        ]).copyProperties(img, ['system:time_start'])

    return s1.map(proc)


def get_lst(aoi, start, end):
    return ee.ImageCollection('MODIS/061/MOD11A1') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('LST_Day_1km')
             .multiply(0.02).subtract(273.15)
             .rename('LST')
             .copyProperties(img, ['system:time_start']))


def get_smap(aoi, start, end):
    return ee.ImageCollection('NASA/SMAP/SPL4SMGP/008') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('sm_surface')
             .rename('SMAP')
             .copyProperties(img, ['system:time_start']))


def get_era5(aoi, start, end):
    col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterBounds(aoi).filterDate(start, end)

    def proc(img):
        return ee.Image.cat([
            img.select("volumetric_soil_water_layer_1").rename("ERA5_SM"),
            img.select("total_precipitation_sum").rename("precip"),
            img.select("surface_net_solar_radiation_sum").divide(1e6).rename("ssr"),
            img.select("temperature_2m").subtract(273.15).rename("T_air"),
            img.select("dewpoint_temperature_2m").subtract(273.15).rename("Td_air")
        ]).copyProperties(img, ["system:time_start"])

    return col.map(proc)


def get_lai(aoi, start, end):
    return ee.ImageCollection("MODIS/061/MCD15A3H") \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select("Lai").multiply(0.1)
             .rename("LAI")
             .copyProperties(img, ['system:time_start']))


def get_topo():
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    return ee.Image.cat([dem.rename('elevation'), slope, aspect])


def get_soil():
    return ee.Image.cat([
        ee.Image("projects/soilgrids-isric/clay_mean")
            .select("clay_0-5cm_mean").rename("clay"),
        ee.Image("projects/soilgrids-isric/sand_mean")
            .select("sand_0-5cm_mean").rename("sand"),
        ee.Image("projects/soilgrids-isric/silt_mean")
            .select("silt_0-5cm_mean").rename("silt")
    ])


# ================= SAFE TEMPORAL FUNCTIONS =================

def safe_median(col, bands):
    return ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            col.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


def get_window(col, date, bands, days):
    start = ee.Date(date).advance(-days, 'day')
    subset = col.filterDate(start, date)  # STRICTLY PAST

    img = ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            safe_median(col, bands)
        )
    )

    return img.select(bands)


def get_daily(col, date, bands):
    subset = col.filterDate(date, ee.Date(date).advance(1, 'day'))

    return ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


# ================= FEATURE BUILDER =================

def get_smap_window(col, date):
    start = ee.Date(date).advance(-6, 'day')
    end = ee.Date(date).advance(1, 'day')

    subset = col.filterDate(start, end)

    return ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.mean(),
            ee.Image.constant([0]).rename(['SMAP']).updateMask(ee.Image(0))
        )
    )


def get_era5_window(col, date):
    start = ee.Date(date).advance(-6, 'day')
    end = ee.Date(date).advance(1, 'day')

    subset = col.filterDate(start, end)

    return ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.mean(),
            ee.Image.constant([0]*5).rename(
                ['ERA5_SM','precip','ssr','T_air','Td_air']
            ).updateMask(ee.Image(0))
        )
    )


def build_stack(date):
    date = ee.Date(date)

    era5 = get_era5_window(era5_col, date)
    smap = get_smap_window(smap_col, date)

    era5_minus_smap = era5.select('ERA5_SM') \
        .subtract(smap.select('SMAP')) \
        .rename('ERA5_minus_SMAP')

    dynamic = ee.Image.cat([
        get_window(s2_col, date, ['NDVI','NDWI'], 30),
        get_window(lai_col, date, ['LAI'], 30),
        get_window(s1_col, date, ['VV','VH','VV_VH_Ratio', 'angle'], 12),
        get_window(lst_col, date, ['LST'], 7),
        era5,
        era5_minus_smap
    ]).resample('bilinear')

    img = ee.Image.cat([
        dynamic,
        smap,
        soil_img,
        topo_img
    ]).reproject(
        crs=TARGET_CRS,
        scale=TARGET_SCALE
    )

    kernel = ee.Kernel.square(radius=PATCH_RADIUS)
    img = img.neighborhoodToArray(kernel)

    return img.set('system:time_start', date.millis())


# ================= EXECUTION =================

BUFFER_START = ee.Date(START_DATE).advance(-30, 'day')

s2_col = get_s2(AOI, BUFFER_START, END_DATE)
s1_col = get_s1(AOI, BUFFER_START, END_DATE)
lai_col = get_lai(AOI, BUFFER_START, END_DATE)
lst_col = get_lst(AOI, START_DATE, END_DATE)
smap_col = get_smap(AOI, START_DATE, END_DATE)
era5_col = get_era5(AOI, START_DATE, END_DATE)

soil_img = get_soil()
topo_img = get_topo()

dates = ee.List.sequence(
    0,
    ee.Date(END_DATE).difference(ee.Date(START_DATE), 'day').subtract(1)
).map(lambda d: ee.Date(START_DATE).advance(d, 'day'))


points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]), {'point_id': r.point_id})
    for r in ground_truth[['point_id','longitude','latitude']].drop_duplicates().itertuples()
])


collection = ee.ImageCollection(dates.map(build_stack))


def sample(img):
    ts = img.get('system:time_start')

    return img.sampleRegions(
        collection=points,
        properties=['point_id'],
        scale=TARGET_SCALE,   # MATCH projection
        tileScale=8,
        geometries=False
    ).map(lambda f: f.set('timestamp', ts))


dataset = collection.map(sample).flatten()


task = ee.batch.Export.table.toDrive(
    collection=dataset,
    description='ML_ready_1km_patches',
    fileFormat='TFRecord'
)

task.start()
print("Export started")

Export started


In [103]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tfrecord.reader import tfrecord_loader
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

torch.manual_seed(42) # Ensure randomness is the same every time
np.random.seed(42)

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [104]:
class HOBEPatchDataset(IterableDataset):
    def __init__(self, data_path, features):
        self.data_path = data_path
        self.features = features
        self.description = {f: "float" for f in features}
        self.description['point_id'] = "byte"
        self.description['timestamp'] = "float"

    def __iter__(self):
        loader = tfrecord_loader(self.data_path, None, self.description, compression_type='gzip')
        for record in loader:
            try:
                patch_list = []
                for feat in self.features:
                    arr = np.array(record[feat])
                    if arr.size != 169:
                        continue  # skip corrupted
                    patch_list.append(arr.reshape(13, 13))

                if len(patch_list) != len(self.features):
                    continue

                x = torch.from_numpy(np.stack(patch_list)).float()
                pid = record['point_id'].decode('utf-8')
                ts = int(record['timestamp'].item())

                yield x, pid, ts

            except:
                continue

In [105]:
features = [
    'NDVI', 'NDWI', 'LAI', 'VV', 'VH', 'VV_VH_Ratio', 
    'LST', 'ERA5_SM', 'precip', 'ssr', 'T_air', 
    'SMAP', 'clay', 'sand', 'silt', 'elevation', 'slope', 'aspect', 'angle', 'ERA5_minus_SMAP'
]
FEATURE_INDEX = {f: i for i, f in enumerate(features)}

def get_anchor(x, i, anchor_type):
    if anchor_type == 'SMAP':
        # spatially stable
        return x[i, FEATURE_INDEX['SMAP']].mean().item()
    
    elif anchor_type == 'ERA5':
        return x[i, FEATURE_INDEX['ERA5_SM']].mean().item()
    
    elif anchor_type == 'HYBRID':
        smap = x[i, FEATURE_INDEX['SMAP']].mean().item()
        era5 = x[i, FEATURE_INDEX['ERA5_SM'], PATCH_CENTER, PATCH_CENTER].item()
        return 0.7 * era5 + 0.3 * smap
    
    else:
        raise ValueError("Unknown anchor")

# Standardize dictionary keys to (str, int_ms)
gt_lookup = {
    (str(row.point_id), int(row.timestamp.value // 10**6)): row.sm 
    for row in ground_truth.itertuples()
}

ds = HOBEPatchDataset("Data/Denmark/ML_ready_1km_patches.tfrecord.gz", features)
loader = DataLoader(ds, batch_size=64, num_workers=0)

print("Caching data to RAM...")
cached_data = []
for x, pids, tss in loader:
    cached_data.append((x, pids, tss))

Caching data to RAM...


In [106]:
all_pids = np.unique([p for _, pids, _ in cached_data for p in pids])

np.random.seed(42)
np.random.shuffle(all_pids)

train_stations = set(all_pids[:int(0.8 * len(all_pids))])
test_stations = set(all_pids[int(0.8 * len(all_pids)):])

In [107]:
def normalize_tensor(x):
    # x: [Batch, Channels, H, W]
    # We normalize each channel independently across the batch
    for c in range(x.shape[1]):
        c_slice = x[:, c, :, :]
        c_min = c_slice.min()
        c_max = c_slice.max()
        
        # If the channel isn't "dead" (max != min)
        if c_max > c_min:
            x[:, c, :, :] = (c_slice - c_min) / (c_max - c_min)
        else:
            # If it's a dead channel (like your 3, 4, 5, 8), keep it at 0
            x[:, c, :, :] = 0.0
    return x


In [108]:
import math

BASE_CHANNELS = len(features)

def add_temporal_features(cached_list, base_channels=BASE_CHANNELS):
    new_cached_data = []
    print("Injecting Day_Sin and Day_Cos features...")

    for x, pids, tss in cached_list:
        if x.shape[1] < base_channels:
            raise ValueError(f"Expected at least {base_channels} channels, got {x.shape[1]}")

        # Always rebuild from original base channels so reruns don't duplicate temporal bands
        x_base = x[:, :base_channels, :, :]
        sin_list = []
        cos_list = []

        for ts in tss:
            dt = pd.to_datetime(ts.item(), unit='ms')
            doy = dt.dayofyear

            sin_val = math.sin(2 * math.pi * doy / 365.25)
            cos_val = math.cos(2 * math.pi * doy / 365.25)

            sin_patch = torch.full((1, 13, 13), sin_val, dtype=x_base.dtype, device=x_base.device)
            cos_patch = torch.full((1, 13, 13), cos_val, dtype=x_base.dtype, device=x_base.device)

            sin_list.append(sin_patch)
            cos_list.append(cos_patch)

        sin_tensor = torch.stack(sin_list)
        cos_tensor = torch.stack(cos_list)

        # Final shape is always [Batch, 20, 13, 13]
        x_new = torch.cat([x_base, sin_tensor, cos_tensor], dim=1)
        new_cached_data.append((x_new, pids, tss))

    return new_cached_data

def get_normalization_stats_from_cache(cached_list):
    all_samples = [x for x, _, _ in cached_list]
    stacked = torch.cat(all_samples, dim=0)
    means = stacked.mean(dim=(0, 2, 3))
    stds = stacked.std(dim=(0, 2, 3)).clamp_min(1e-6)
    return means, stds

cached_data = add_temporal_features(cached_data)

Injecting Day_Sin and Day_Cos features...


In [109]:
def add_precip_features(cached_list):
    print("Adding proper precip accumulation features...")

    # Flatten everything first for temporal ordering
    records = []
    for x, pids, tss in cached_list:
        for i in range(len(pids)):
            records.append((pids[i], int(tss[i].item()), x[i]))

    # Group by station
    from collections import defaultdict
    station_dict = defaultdict(list)

    for pid, ts, x in records:
        station_dict[pid].append((ts, x))

    # Sort temporally
    for pid in station_dict:
        station_dict[pid].sort(key=lambda v: v[0])

    # Build new dataset
    new_records = []

    for pid, seq in station_dict.items():
        for i in range(len(seq)):
            ts, x = seq[i]

            def accumulate(days):
                total = torch.zeros_like(seq[i][1][FEATURE_INDEX['precip']])
                for j in range(max(0, i - days + 1), i + 1):
                    total = total + seq[j][1][FEATURE_INDEX['precip']]
                return total

            p3 = accumulate(3)
            p7 = accumulate(7)

            new_x = torch.cat([
                x,
                p3.unsqueeze(0),
                p7.unsqueeze(0)
            ], dim=0)

            new_records.append((new_x, pid, ts))

    # Re-batch
    batched = []
    B = 64
    for i in range(0, len(new_records), B):
        batch = new_records[i:i+B]
        x = torch.stack([r[0] for r in batch])
        p = [r[1] for r in batch]
        t = torch.tensor([r[2] for r in batch])
        batched.append((x, p, t))

    return batched

cached_data = add_precip_features(cached_data)

Adding proper precip accumulation features...


In [110]:
def get_train_stats(cached_list):
    xs = []
    for x, pids, _ in cached_list:
        mask = [i for i, p in enumerate(pids) if p in train_stations]
        if mask:
            xs.append(x[mask])
    stacked = torch.cat(xs, dim=0)

    means = stacked.mean(dim=(0,2,3))
    stds = stacked.std(dim=(0,2,3)).clamp_min(1e-6)

    return means, stds

means, stds = get_train_stats(cached_data)
means_gpu = means.view(1, -1, 1, 1).to(device)
stds_gpu = stds.view(1, -1, 1, 1).to(device)
means_gpu = means.view(1, -1, 1, 1).to(device)
stds_gpu = stds.view(1, -1, 1, 1).to(device)

print(f"Updated channel count after temporal features: {means.numel()}")

Updated channel count after temporal features: 24


In [111]:
FINAL_CHANNELS = cached_data[0][0].shape[1]
print("Final channels:", FINAL_CHANNELS)

Final channels: 24


In [112]:
class SoilCNN(nn.Module):
    def __init__(self, in_channels=22):
        super(SoilCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # Added
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64) # Added
        self.drop = nn.Dropout2d(0.3) 
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0) 
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = nn.Linear(128 * 3 * 3, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.drop(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SoilCNN(in_channels=FINAL_CHANNELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4) 
criterion = nn.SmoothL1Loss(beta=0.02)

In [113]:
# Check residual scale
residuals = []
for (pid, ts), sm in gt_lookup.items():
    # find one sample
    for x, pids, tss in cached_data:
        for i in range(len(pids)):
            if pids[i] == pid:
                smap = x[i, FEATURE_INDEX['SMAP'], PATCH_CENTER, PATCH_CENTER].item()
                era5 = x[i, FEATURE_INDEX['ERA5_SM'], PATCH_CENTER, PATCH_CENTER].item()
                residuals.append(sm - era5)
                break

print("Residual stats:")
print(np.mean(residuals), np.std(residuals), np.min(residuals), np.max(residuals))

Residual stats:
-0.05319633005193772 0.08763871672034805 -0.32274257453282673 0.3415789693991343


In [114]:
print(f"Starting Training: {len(train_stations)} train, {len(test_stations)} test stations.")

for epoch in range(15):
    model.train()
    total_loss, match_count = 0, 0

    for x, pids, tss in cached_data:
        y_list, valid_indices = [], []

        for i in range(len(pids)):
            if pids[i] in train_stations:
                floored_ts = int((tss[i].item() // 86400000) * 86400000)
                label = gt_lookup.get((pids[i], floored_ts))

                if label is not None and not np.isnan(label):
                    smap_val = x[i, FEATURE_INDEX['SMAP'], PATCH_CENTER, PATCH_CENTER].item()
                    era5_val = x[i, FEATURE_INDEX['ERA5_SM'],PATCH_CENTER, PATCH_CENTER].item()
                    residual = label - era5_val

                    y_list.append(residual)
                    valid_indices.append(i)

        if not y_list:
            continue

        x_raw = x[valid_indices].to(device)

        # keep SMAP in original scale
        smap_batch = x_raw[:, FEATURE_INDEX['SMAP'], PATCH_CENTER, PATCH_CENTER].unsqueeze(1)
        era5_batch = x_raw[:, FEATURE_INDEX['ERA5_SM'], PATCH_CENTER, PATCH_CENTER].unsqueeze(1)

        # normalize inputs
        x_batch = (x_raw - means_gpu) / (stds_gpu + 1e-6)

        y_batch = torch.tensor(y_list).float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        pred_residual = model(x_batch)

        loss = criterion(pred_residual, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_list)
        match_count += len(y_list)

    print(f"Epoch {epoch} | Train Loss: {total_loss / match_count:.6f}")

Starting Training: 24 train, 6 test stations.
Epoch 0 | Train Loss: 0.048278
Epoch 1 | Train Loss: 0.048381
Epoch 2 | Train Loss: 0.047594
Epoch 3 | Train Loss: 0.044531
Epoch 4 | Train Loss: 0.039973
Epoch 5 | Train Loss: 0.034120
Epoch 6 | Train Loss: 0.030178
Epoch 7 | Train Loss: 0.026300
Epoch 8 | Train Loss: 0.024501
Epoch 9 | Train Loss: 0.023293
Epoch 10 | Train Loss: 0.022047
Epoch 11 | Train Loss: 0.021575
Epoch 12 | Train Loss: 0.020497
Epoch 13 | Train Loss: 0.020256
Epoch 14 | Train Loss: 0.019573


In [115]:
model.eval()
preds, actuals = [], []

with torch.no_grad():
    for x, pids, tss in cached_data:
        for i in range(len(pids)):
            if pids[i] in test_stations:
                floored_ts = int((tss[i].item() // 86400000) * 86400000)
                label = gt_lookup.get((pids[i], floored_ts))

                if label is not None and not np.isnan(label):
                    base = x[i].unsqueeze(0)  # [1, 22, 13, 13]

                    x_input = x[i].unsqueeze(0).to(device)

                    # SAME normalization as training
                    x_input = (x_input - means_gpu) / (stds_gpu + 1e-6)

                    residual = model(x_input).item()
                    smap_val = x[i, FEATURE_INDEX['SMAP'], PATCH_CENTER, PATCH_CENTER].item()
                    era5_val = x[i, FEATURE_INDEX['ERA5_SM'], PATCH_CENTER, PATCH_CENTER].item()
                    pred = residual + era5_val

                    preds.append(pred)
                    actuals.append(label)


print(f"Unseen Station R2 Score: {r2_score(actuals, preds):.4f}")

Unseen Station R2 Score: 0.2890


In [116]:
print("\n=== CHANNEL STATS ===")
sample_x, _, _ = cached_data[0]

for i in range(sample_x.shape[1]):
    mean = sample_x[:, i].mean().item()
    std = sample_x[:, i].std().item()
    print(f"Channel {i}: mean={mean:.3f}, std={std:.3f}")


=== CHANNEL STATS ===
Channel 0: mean=0.535, std=0.006
Channel 1: mean=0.171, std=0.000
Channel 2: mean=1.290, std=0.355
Channel 3: mean=-12.670, std=1.346
Channel 4: mean=-17.826, std=0.422
Channel 5: mean=4.862, std=0.845
Channel 6: mean=13.160, std=3.169
Channel 7: mean=0.270, std=0.024
Channel 8: mean=0.003, std=0.002
Channel 9: mean=14.151, std=2.036
Channel 10: mean=8.106, std=1.882
Channel 11: mean=0.211, std=0.015
Channel 12: mean=58.101, std=6.397
Channel 13: mean=842.154, std=16.171
Channel 14: mean=100.467, std=12.885
Channel 15: mean=63.396, std=9.596
Channel 16: mean=0.249, std=0.176
Channel 17: mean=211.025, std=85.312
Channel 18: mean=36.129, std=0.655
Channel 19: mean=0.031, std=0.015
Channel 20: mean=0.829, std=0.155
Channel 21: mean=-0.464, std=0.270
Channel 22: mean=0.008, std=0.005
Channel 23: mean=0.017, std=0.010


In [117]:
dead_channels = (stds < 1e-5).nonzero().flatten().tolist()
print("Dead channels:", dead_channels)

Dead channels: []


In [118]:
def check_tensor(x):
    if torch.isnan(x).any():
        print("NaNs detected")
    if torch.isinf(x).any():
        print("Infs detected")

for x, _, _ in cached_data[:5]:
    check_tensor(x)

In [119]:
vals = list(gt_lookup.values())
print("\nTarget stats:")
print(f"Mean: {np.mean(vals):.4f}, Std: {np.std(vals):.4f}")
print(f"Min: {np.min(vals):.4f}, Max: {np.max(vals):.4f}")


Target stats:
Mean: 0.2240, Std: 0.0808
Min: 0.0000, Max: 0.4940


In [120]:
print(f"Train stations: {len(train_stations)}")
print(f"Test stations: {len(test_stations)}")
print(f"Total samples: {sum(len(p) for _, p, _ in cached_data)}")

Train stations: 24
Test stations: 6
Total samples: 30870


In [121]:
print("\n=== FEATURE ↔ TARGET CORR ===")
for i, f in enumerate(features):
    vals = []
    targets = []
    for x, pids, tss in cached_data:
        for j in range(len(pids)):
            key = (pids[j], int((tss[j].item() // 86400000) * 86400000))
            if key in gt_lookup:
                vals.append(x[j, i, PATCH_CENTER, PATCH_CENTER].item())
                targets.append(gt_lookup[key])
    if len(vals) > 0:
        corr = np.corrcoef(vals, targets)[0,1]
        print(f"{f}: {corr:.3f}")


=== FEATURE ↔ TARGET CORR ===
NDVI: -0.071
NDWI: 0.054
LAI: -0.285
VV: -0.061
VH: -0.102
VV_VH_Ratio: -0.015
LST: -0.311
ERA5_SM: 0.325
precip: 0.087
ssr: -0.283
T_air: -0.283
SMAP: 0.289
clay: 0.119
sand: -0.078
silt: 0.024
elevation: 0.095
slope: 0.190
aspect: 0.010
angle: -0.026
ERA5_minus_SMAP: 0.062
